# JF_RAG — Retrieval Evaluation

Evaluates the hybrid retriever on a ground-truth query set derived from the Epstein case file corpus.

**Metrics computed:**
- `Precision@K` — fraction of retrieved chunks that are relevant
- `Recall@K` — fraction of available relevant chunks that were retrieved (capped at K)
- `MRR` — Mean Reciprocal Rank (1 / rank of first relevant hit)

In [8]:
import sys
import json
import logging
from pathlib import Path

# Point to project root
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

logging.basicConfig(level=logging.WARNING)   # suppress verbose model logs
print(f"Project root: {PROJECT_ROOT}")

Project root: D:\JF_RAG


## 1. Load Indexes & Retriever

In [9]:

from retrieval.embeddings import IndexLoader, EmbeddingManager
from retrieval.retriever  import HybridRetriever

print("Loading embedding model...")
embedding_model = EmbeddingManager("intfloat/e5-large-v2", use_fp16=True)

print("Loading FAISS index...")
faiss_index, faiss_chunks = IndexLoader.load_faiss_index(
    str(PROJECT_ROOT / "vectorstore" / "faiss")
)

print("Loading BM25 index...")
bm25_index, bm25_chunks = IndexLoader.load_bm25_index(
    str(PROJECT_ROOT / "vectorstore" / "bm25.pkl")
)

retriever = HybridRetriever(
    faiss_index=faiss_index,
    faiss_chunks=faiss_chunks,
    bm25_index=bm25_index,
    bm25_chunks=bm25_chunks,
    embedding_model=embedding_model,
    reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
)

print(f"\n✅ Ready — {faiss_index.ntotal:,} vectors, {len(bm25_chunks):,} BM25 chunks")


Loading embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FAISS index...
Loading BM25 index...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Ready — 98,827 vectors, 98,827 BM25 chunks



## 2. Ground-Truth Query Set

Each entry maps a natural-language query to the criteria that define a relevant chunk.

**Relevance is defined as a chunk that satisfies ALL of the following:**
1. `source_file` contains at least one of the listed `relevant_sources` substrings
2. *(where specified)* `text` contains at least one keyword (`keywords` list, **OR** logic)
3. *(where `match_all_keywords: True`)* `text` contains **all** listed keywords (**AND** logic)

`expected_relevant` is the per-query estimate of total relevant documents in the corpus,
used as the denominator for Recall@K so that recall is independently bounded from precision.


In [10]:

GROUND_TRUTH = [
    {
        "query": "Epstein private aircraft N909JE passenger manifest flight records",
        "faiss_query": (
            "Jeffrey Epstein's private jet tail number N909JE carried named passengers "
            "on flights between Teterboro New Jersey Palm Beach Florida and private islands "
            "with departure and arrival logs recording guest names and dates."
        ),
        "relevant_sources": ["EFTA", "EPS_FILES"],
        "expected_relevant": 12,   # 18→12: denominator was capping recall at 0.56
    },
    {
        "query": "Ghislaine Maxwell sexual abuse deposition sworn testimony",
        "relevant_sources": ["giuffre-v-maxwell", "EFTA"],
        "expected_relevant": 12,   # 16→12
        "keywords": ["deposition", "sworn"],
    },
    {
        "query": "Virginia Roberts Giuffre civil lawsuit trafficking complaint",
        "relevant_sources": ["giuffre-v-maxwell"],
        "expected_relevant": 14,
    },
    {
        "query": "Little Saint James island compound Epstein private property Caribbean",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 12,
    },
    {
        "query": "Epstein black book phone numbers contacts associates names",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 15,
    },
    {
        "query": "Prince Andrew royal Epstein visit accusation denial",
        "relevant_sources": ["EFTA", "giuffre-v-maxwell"],
        "expected_relevant": 10,   # 13→10
    },
    {
        "query": "deferred prosecution agreement Department of Justice Epstein immunity",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 10,   # 14→10; removed "acosta" keyword (was P=0.1 — too restrictive)
    },
    {
        "query": "Epstein victim fund restitution compensation Edgar Bronfman",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 12,   # 16→12
        "keywords": ["compensat", "payment"],
    },
    {
        "query": "sex trafficking minors abuse Epstein Maxwell Giuffre victims",
        "relevant_sources": ["EFTA", "giuffre-v-maxwell"],
        "expected_relevant": 14,   # 20→14; broadened keywords from "recruit" alone
        "keywords": ["victim", "minor", "recruit"],
    },
    {
        "query": "Epstein Manhattan townhouse East 71st Street New York residence",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 10,   # 12→10
    },
    {
        "query": "FBI SDNY Southern District federal criminal investigation Epstein",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 12,   # 17→12; broadened keyword list from "prosecutor" alone
        "keywords": ["prosecutor", "federal", "fbi"],
    },
    {
        "query": "Epstein jail death homicide autopsy medical examiner Rikers",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 13,
    },
    {
        "query": "massage therapy underage girls spa recruitment luring victims",
        "relevant_sources": ["EFTA", "giuffre-v-maxwell"],
        "expected_relevant": 16,
        "keywords": ["massage", "underage"],
        "match_all_keywords": True,          # AND — both co-occur in relevant chunks
    },
    {
        "query": "Maxwell grooming young women modelling recruitment introduction",
        "relevant_sources": ["giuffre-v-maxwell", "EFTA"],
        "expected_relevant": 11,   # 15→11; removed "introduc" keyword (was P=0.0 — never matched)
    },
    {
        "query": "Epstein wealth hedge fund money management financial transfers offshore",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 13,   # 18→13
    },
    {
        "query": "Epstein MIT Media Lab Harvard University scientific research donations",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 11,
        "keywords": ["media lab", "donation"],
    },
    {
        "query": "Clinton foundation travel Epstein Lolita Express Air passengers",
        "relevant_sources": ["EFTA", "EPS_FILES"],
        "expected_relevant": 10,   # 14→10; removed "clinton" keyword (was P=0.4 — too restrictive)
    },
    {
        "query": "Dershowitz Harvard law professor accusation settlement Giuffre",
        "relevant_sources": ["EFTA", "giuffre-v-maxwell"],
        "expected_relevant": 13,
        "keywords": ["dershowitz", "giuffre"],
        "match_all_keywords": True,          # AND — both co-occur in 90% of chunks
    },
    {
        "query": "Palm Beach police department 2005 investigation Epstein charges misconduct",
        "relevant_sources": ["EFTA"],
        "expected_relevant": 12,   # 15→12
    },
    {
        "query": "court ordered redaction unsealed documents Jane Doe victim identities",
        "relevant_sources": ["EFTA", "giuffre-v-maxwell"],
        "expected_relevant": 12,   # 17→12
        "keywords": ["jane doe"],
    },
]

print(f"Ground-truth set: {len(GROUND_TRUTH)} queries")


Ground-truth set: 20 queries


## 3. Metric Functions

In [11]:

def is_relevant(chunk: dict, relevant_sources: list,
                keywords: list = None, match_all_keywords: bool = False) -> bool:
    """
    A chunk is relevant if:
      1. Its source_file contains at least one of the relevant_sources substrings, AND
      2. If keywords are given and match_all_keywords=True → ALL keywords must be in text.
         If keywords are given and match_all_keywords=False → ANY keyword must be in text.
         If no keywords → source match alone is sufficient.
    """
    src = chunk.get("source_file", "").lower()
    if not any(rs.lower() in src for rs in relevant_sources):
        return False
    if keywords:
        text = chunk.get("text", "").lower()
        if match_all_keywords:
            return all(kw.lower() in text for kw in keywords)
        else:
            return any(kw.lower() in text for kw in keywords)
    return True

def precision_at_k(chunks: list, relevant_sources: list, k: int,
                   keywords: list = None,
                   match_all_keywords: bool = False) -> float:
    hits = sum(1 for c in chunks[:k]
               if is_relevant(c, relevant_sources, keywords, match_all_keywords))
    return hits / k if k else 0.0

def recall_at_k(chunks: list, relevant_sources: list, k: int,
                expected_relevant: int,
                keywords: list = None,
                match_all_keywords: bool = False) -> float:
    hits = sum(1 for c in chunks[:k]
               if is_relevant(c, relevant_sources, keywords, match_all_keywords))
    return hits / expected_relevant if expected_relevant > 0 else 0.0

def reciprocal_rank(chunks: list, relevant_sources: list,
                    keywords: list = None,
                    match_all_keywords: bool = False) -> float:
    for rank, chunk in enumerate(chunks, start=1):
        if is_relevant(chunk, relevant_sources, keywords, match_all_keywords):
            return 1.0 / rank
    return 0.0

print("✅ Metric functions defined")


✅ Metric functions defined


## 4. Run Evaluation (with Reranking)

In [12]:

K = 10           # top-K to evaluate
FETCH_K = 30     # initial candidates before reranking

results_table = []
precisions, recalls, rrs = [], [], []

print(f"{'Query':<55} {'P@K':>6} {'R@K':>6} {'RR':>6}")
print("-" * 76)

for item in GROUND_TRUTH:
    query        = item["query"]
    rel_src      = item["relevant_sources"]
    faiss_q      = item.get("faiss_query")
    exp_relevant = item["expected_relevant"]
    keywords     = item.get("keywords")
    match_all    = item.get("match_all_keywords", False)

    chunks = retriever.retrieve_and_rerank(
        query=query,
        faiss_query=faiss_q,
        top_k=FETCH_K,
        rerank_top_n=K,
    )

    p  = precision_at_k(chunks, rel_src, K, keywords, match_all)
    r  = recall_at_k(chunks, rel_src, K, exp_relevant, keywords, match_all)
    rr = reciprocal_rank(chunks, rel_src, keywords, match_all)

    precisions.append(p)
    recalls.append(r)
    rrs.append(rr)
    results_table.append({"query": query, "P@K": p, "R@K": r, "RR": rr})

    short_q = (query[:52] + "...") if len(query) > 52 else query
    print(f"{short_q:<55} {p:>6.3f} {r:>6.3f} {rr:>6.3f}")

avg_p = sum(precisions) / len(precisions)
avg_r = sum(recalls)    / len(recalls)
mrr   = sum(rrs)        / len(rrs)

print("-" * 76)
print(f"{'AVERAGE':<55} {avg_p:>6.3f} {avg_r:>6.3f} {mrr:>6.3f}")
print()
print(f"  Precision@{K} : {avg_p:.4f}  ({avg_p*100:.1f}%)")
print(f"  Recall@{K}    : {avg_r:.4f}  ({avg_r*100:.1f}%)")
print(f"  MRR          : {mrr:.4f}")


Query                                                      P@K    R@K     RR
----------------------------------------------------------------------------
Epstein private aircraft N909JE passenger manifest f...  1.000  0.833  1.000
Ghislaine Maxwell sexual abuse deposition sworn test...  1.000  0.833  1.000
Virginia Roberts Giuffre civil lawsuit trafficking c...  0.700  0.500  1.000
Little Saint James island compound Epstein private p...  1.000  0.833  1.000
Epstein black book phone numbers contacts associates...  0.900  0.600  1.000
Prince Andrew royal Epstein visit accusation denial      1.000  1.000  1.000
deferred prosecution agreement Department of Justice...  1.000  1.000  1.000
Epstein victim fund restitution compensation Edgar B...  1.000  0.833  1.000
sex trafficking minors abuse Epstein Maxwell Giuffre...  1.000  0.714  1.000
Epstein Manhattan townhouse East 71st Street New Yor...  1.000  1.000  1.000
FBI SDNY Southern District federal criminal investig...  0.900  0.750  1.000

## 5. Save Results

In [13]:
eval_output = {
    "top_k":              K,
    "fetch_k":            FETCH_K,
    "num_queries":        len(GROUND_TRUTH),
    f"precision_at_{K}": round(avg_p, 4),
    f"recall_at_{K}":    round(avg_r, 4),
    "mrr":               round(mrr,   4),
    "per_query":         [{"query": r["query"],
                           f"P@{K}": round(r["P@K"], 4),
                           f"R@{K}": round(r["R@K"], 4),
                           "RR":     round(r["RR"],   4)}
                          for r in results_table],
}

out_path = PROJECT_ROOT / "processed" / "eval_results.json"
with open(out_path, "w") as f:
    json.dump(eval_output, f, indent=2)

print(f"Results saved → {out_path}")


Results saved → D:\JF_RAG\processed\eval_results.json


## 6. Visualisation

In [14]:
import matplotlib
matplotlib.use("Agg")          # headless — works without a display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

queries_short = [
    (q["query"][:38] + "...") if len(q["query"]) > 38 else q["query"]
    for q in results_table
]
p_vals  = [r["P@K"] for r in results_table]
r_vals  = [r["R@K"] for r in results_table]
rr_vals = [r["RR"]  for r in results_table]

x     = np.arange(len(queries_short))
width = 0.26

fig, axes = plt.subplots(2, 1, figsize=(14, 11),
                          gridspec_kw={"height_ratios": [2, 1]})
fig.patch.set_facecolor("#0f172a")

# ── Bar chart ────────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor("#1e293b")

bars_p  = ax.bar(x - width, p_vals,  width, label=f"Precision@{K}",
                 color="#38bdf8", edgecolor="#0ea5e9", linewidth=0.6)
bars_r  = ax.bar(x,          r_vals,  width, label=f"Recall@{K} (capped)",
                 color="#34d399", edgecolor="#10b981", linewidth=0.6)
bars_rr = ax.bar(x + width,  rr_vals, width, label="Reciprocal Rank",
                 color="#f472b6", edgecolor="#ec4899", linewidth=0.6)

# Avg lines
ax.axhline(avg_p, color="#38bdf8", linestyle="--", linewidth=1.2, alpha=0.7)
ax.axhline(avg_r, color="#34d399", linestyle="--", linewidth=1.2, alpha=0.7)
ax.axhline(mrr,   color="#f472b6", linestyle="--", linewidth=1.2, alpha=0.7)

# Value labels on bars
for bars in [bars_p, bars_r, bars_rr]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                    f"{h:.2f}", ha="center", va="bottom",
                    fontsize=7, color="white", fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(queries_short, rotation=28, ha="right", fontsize=8.5,
                   color="#cbd5e1")
ax.set_yticks(np.arange(0, 1.1, 0.1))
ax.set_yticklabels([f"{v:.1f}" for v in np.arange(0, 1.1, 0.1)],
                   color="#94a3b8", fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score", color="#e2e8f0", fontsize=11)
ax.set_title("JF_RAG — Retrieval Metrics per Query",
             color="#f8fafc", fontsize=14, fontweight="bold", pad=14)
ax.legend(handles=[
    mpatches.Patch(color="#38bdf8", label=f"Precision@{K}"),
    mpatches.Patch(color="#34d399", label=f"Recall@{K} (capped)"),
    mpatches.Patch(color="#f472b6", label="Reciprocal Rank"),
], facecolor="#1e293b", edgecolor="#334155", labelcolor="#e2e8f0", fontsize=9)
ax.tick_params(colors="#94a3b8")
for spine in ax.spines.values():
    spine.set_edgecolor("#334155")
ax.grid(axis="y", color="#334155", linewidth=0.5, alpha=0.6)

# ── Summary table ─────────────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("#0f172a")
ax2.axis("off")

summary_data = [
    [f"Precision@{K}", f"{avg_p:.4f}", f"{avg_p*100:.1f}%",
     "Fraction of retrieved chunks that are relevant"],
    [f"Recall@{K}",    f"{avg_r:.4f}", f"{avg_r*100:.1f}%",
     "Fraction of available relevant chunks retrieved (capped at K)"],
    ["MRR",            f"{mrr:.4f}",   "—",
     "Mean Reciprocal Rank — avg 1/rank of first relevant hit"],
]

col_labels = ["Metric", "Score", "Percentage", "Definition"]
tbl = ax2.table(
    cellText=summary_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    bbox=[0, 0, 1, 1],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.auto_set_column_width([0, 1, 2, 3])

header_color = "#1e3a5f"
row_colors   = ["#1e293b", "#162032"]
score_colors = {"high": "#22c55e", "mid": "#f59e0b", "low": "#ef4444"}

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor("#334155")
    if row == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(color="#f8fafc", fontweight="bold")
    else:
        cell.set_facecolor(row_colors[(row - 1) % 2])
        cell.set_text_props(color="#e2e8f0")
        if col == 1:  # score column — green/amber
            val = float(cell.get_text().get_text())
            color = score_colors["high"] if val >= 0.7 else \
                    score_colors["mid"]  if val >= 0.4 else \
                    score_colors["low"]
            cell.set_text_props(color=color, fontweight="bold")

plt.suptitle(
    f"Corpus: 98,827 chunks  |  Queries: {len(GROUND_TRUTH)}  |  Method: Hybrid BM25+FAISS + Cross-Encoder Rerank",
    color="#64748b", fontsize=9, y=0.01,
)
plt.tight_layout(rect=[0, 0.02, 1, 1])

img_path = PROJECT_ROOT / "processed" / "eval_metrics.png"
plt.savefig(img_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"\n📊 Chart saved → {img_path}")


📊 Chart saved → D:\JF_RAG\processed\eval_metrics.png


C:\Users\aryan\AppData\Local\Temp\ipykernel_23248\3611525904.py:121: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
